# 0 - Imports

In [1]:
from __future__ import annotations
import tensorflow as tf

tf.config.optimizer.set_jit(False)

for gpu in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
        print("TF config OK")
    except RuntimeError:
        pass


I0000 00:00:1779401452.936559    8512 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779401453.135095    8512 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779401454.697133    8512 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
import os
import gc
import multiprocessing as mp
import json
import time
import sys
import numpy as np
from scipy.stats import mannwhitneyu
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import keras
import psutil
#!pip install mealpy
#!pip install optuna
from mealpy import FloatVar, IntegerVar
from mealpy.swarm_based.PSO import AIW_PSO
from mealpy.evolutionary_based.GA import BaseGA
from mealpy.evolutionary_based.DE import SADE
from mealpy.swarm_based.BSA import OriginalBSA
from mealpy.swarm_based.GWO import OriginalGWO
import optuna
import optuna.visualization as vis
from optuna.visualization import matplotlib as ovm
from optuna.importance import get_param_importances, FanovaImportanceEvaluator
from keras import layers, models
from keras.optimizers import SGD, AdamW, RMSprop
optuna.logging.set_verbosity(optuna.logging.WARNING)

os.makedirs('results/irace',         exist_ok=True)
os.makedirs('results/pso_final',     exist_ok=True)


In [3]:
_PROC = psutil.Process(os.getpid())
def _hostRamGB():
    """Resident Set Size of the current Python process, in GB."""
    return _PROC.memory_info().rss / 1024**3

# 1 - Datasets

In [4]:
CIFAR10_CLASSES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

MNIST_CLASSES = [
    '0', '1', '2', '3', '4',
    '5', '6', '7', '8', '9'
]


def loadCifar10(val_split=0.2, seed=42):
    """
    Carrega e pre-processa o dataset CIFAR-10.

    Parametros:
        val_split (float): Fracao do treino usada para validacao. Default: 0.2
        seed (int)       : Seed para reprodutibilidade. Default: 42

    Retorna:
        dict com:
            x_train, y_train         — imagens normalizadas + labels one-hot
            x_test,  y_test          — dados de teste
            class_names              — lista dos 10 nomes de classes
            num_classes              — 10
            input_shape              — (32, 32, 3)
    """
    num_classes = 10

    # 50 000 imagens de treino + 10 000 de teste (split oficial)
    (x_train_full, y_train_full), (x_test, y_test) = keras.datasets.cifar10.load_data()

    # Normalizar pixeis para [0, 1]
    x_train_full = x_train_full.astype('float32') / 255.0
    x_test       = x_test.astype('float32')       / 255.0


    y_train_cat = keras.utils.to_categorical(y_train_full, num_classes)
    y_test_cat  = keras.utils.to_categorical(y_test,       num_classes)

    print(f"  CIFAR-10 loaded:")
    print(f"    Train      : {x_train_full.shape[0]:>6} samples")
    print(f"    Test       : {x_test.shape[0]:>6} samples")
    print(f"    Shape      : {x_train_full.shape[1:]}")
    print(f"    Classes    : {CIFAR10_CLASSES}")

    return {
        'x_train':     x_train_full,
        'y_train':     y_train_cat,
        'x_test':      x_test,
        'y_test':      y_test_cat,
        'class_names': CIFAR10_CLASSES,
        'num_classes': num_classes,
        'input_shape': x_train_full.shape[1:],   # (32, 32, 3)
    }


def loadMnist(val_split=0.2, seed=42):
    """
    Carrega e pre-processa o dataset MNIST.

    Parametros:
        val_split (float): Fracao do treino usada para validacao. Default: 0.2
        seed (int)       : Seed para reprodutibilidade. Default: 42

    Retorna:
        dict com:
            x_train, y_train         — imagens normalizadas + labels one-hot
            x_test,  y_test          — dados de teste
            class_names              — lista dos 10 nomes de classes (digitos 0-9)
            num_classes              — 10
            input_shape              — (28, 28, 1)
    """
    num_classes = 10

    # 60 000 imagens de treino + 10 000 de teste (split oficial)
    (x_train_full, y_train_full), (x_test, y_test) = keras.datasets.mnist.load_data()

    # Normalizar pixeis para [0, 1]
    x_train_full = x_train_full.astype('float32') / 255.0
    x_test       = x_test.astype('float32')       / 255.0

    # Adicionar dimensao do canal: (N, 28, 28) -> (N, 28, 28, 1)
    x_train_full = np.expand_dims(x_train_full, axis=-1)
    x_test       = np.expand_dims(x_test,       axis=-1)

    y_train_cat = keras.utils.to_categorical(y_train_full, num_classes)
    y_test_cat  = keras.utils.to_categorical(y_test,       num_classes)

    print(f"  MNIST loaded:")
    print(f"    Train      : {x_train_full.shape[0]:>6} samples")
    print(f"    Test       : {x_test.shape[0]:>6} samples")
    print(f"    Shape      : {x_train_full.shape[1:]}")
    print(f"    Classes    : {MNIST_CLASSES}")

    return {
        'x_train':     x_train_full,
        'y_train':     y_train_cat,
        'x_test':      x_test,
        'y_test':      y_test_cat,
        'class_names': MNIST_CLASSES,
        'num_classes': num_classes,
        'input_shape': x_train_full.shape[1:],   # (28, 28, 1)
    }


# ─────────────────────────────────────────────────────────────────────────────
# Factory — adiciona novos datasets aqui no futuro
# ─────────────────────────────────────────────────────────────────────────────

_DATASET_REGISTRY = {
    'cifar10': loadCifar10,
    'mnist':   loadMnist,
    # 'cifar100': load_cifar100,   # futuro
    # 'imagenet': load_imagenet,   # futuro
}


def getDataset(name='cifar10', **kwargs):
    """
    Carrega um dataset pelo nome.

    Parametros:
        name (str) : Nome do dataset. Disponiveis: ['cifar10', 'mnist']
        **kwargs   : Argumentos passados ao loader (val_split, seed, ...)

    Retorna:
        dict com os dados prontos para treino/teste.

    Exemplo:
        data = getDataset('cifar10', val_split=0.2, seed=42)
        data['x_train'].shape   # (40000, 32, 32, 3)
    """
    name = name.lower()
    if name not in _DATASET_REGISTRY:
        raise ValueError(
            f"Dataset '{name}' not available. "
            f"Availables: {list(_DATASET_REGISTRY.keys())}"
        )
    return _DATASET_REGISTRY[name](**kwargs)


def listDatasets():
    """Retorna lista dos datasets disponiveis."""
    return list(_DATASET_REGISTRY.keys())

# 2 - Models

In [5]:
def _getActivationLayer(activation):
    if activation == 'leaky_relu':
        return layers.LeakyReLU()
    return layers.Activation(activation)


def _getOptimizer(name, learning_rate):
    name   = name.lower()
    if name == 'sgd':
        return SGD(learning_rate=learning_rate, nesterov=True)
    if name == 'rmsprop':
        return RMSprop(learning_rate=learning_rate)    
    return AdamW(learning_rate=learning_rate)


def buildTemplateCNN(config, input_shape, num_classes):
    k       = config['kernel_size']
    pool_fn = layers.AveragePooling2D if config['pooling'] == 'average' else layers.MaxPooling2D
    base    = config['base_filters']

    # VGG doubling rule — design principle, NOT a search dimension.
    # With n_blocks=3 and base=64 -> [64, 128, 256].
    n_blocks = config['n_blocks']
    filters = [base * (2 ** i) for i in range(n_blocks)]

    model = models.Sequential(name='Template_CNN')
    model.add(layers.Input(shape=input_shape))

    for n_filters in filters:
        # Double conv per block (VGG-style)
        model.add(layers.Conv2D(n_filters, (k, k), padding='same'))
        model.add(layers.BatchNormalization())
        model.add(_getActivationLayer(config['activation']))

        model.add(layers.Conv2D(n_filters, (k, k), padding='same'))
        model.add(layers.BatchNormalization())
        model.add(_getActivationLayer(config['activation']))

        model.add(pool_fn((2, 2)))

    # Classifier head (also topologically fixed)
    model.add(layers.Flatten())
    model.add(layers.Dense(config['dense_units']))
    model.add(_getActivationLayer(config['activation']))
    if config['dropout_rate'] > 0:
        model.add(layers.Dropout(config['dropout_rate']))
    model.add(layers.Dense(num_classes, activation='softmax'))

    model.compile(
        optimizer=_getOptimizer(
            config['optimizer'],
            config['learning_rate'],
        ),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

# 3 - Mealpy Optimization Algorithms

In [6]:
ALGORITHM_INFO = {
    'AIW_PSO': 'Adaptive Inertia Weight PSO       — w adapts per-particle from fitness rank (Qin et al., 2006)',
    'GA':  'Genetic Algorithm                 — selection, crossover, mutation',
    'SADE':  'Self-Adaptive Differential Evolution (SADE) — F/CR and strategy adapt from success history (Qin & Suganthan, 2005)',
    'BSA': 'Backtracking Search Algorithm     — mutation with historical population memory',
    'GWO': 'Grey Wolf Optimizer               — wolf hierarchy',
}

_LITERATURE_DEFAULTS = {
    # qin2006adaptive
    'PSO': {
        'params': {'c1': 2.05, 'c2': 2.05, 'alpha': 0.4},
        'pop_size': 30,
    },
    # goldberg1989genetic, dejong1975analysis, grefenstette1986optimization
    'GA': {
        'params': {
            'pc': 0.8, 'pm': 0.01,
            'selection': 'tournament', 'k_way': 0.2,
            'crossover': 'one_point',
            'mutation_multipoints': False, 'mutation': 'flip',
        },
        'pop_size': 50,
    },
    # qin2005self
    'DE': {
        'params': {},
        'pop_size': 50,
    },
    # civicioglu2013backtracking
    'BSA': {
        'params': {'ff': 10, 'pff': 0.8, 'c1': 2.0, 'c2': 2.0, 'a1': 1.5, 'a2': 1.5, 'fc': 0.5},
        'pop_size': 30,
    },
    # mirjalili2014grey
    'GWO': {
        'params': {},
        'pop_size': 30,
    },
}

def getLiteratureDefaults(algo_name):
    name = algo_name.upper()
    if name not in _LITERATURE_DEFAULTS:
        raise ValueError(f"Algorithm '{name}' not found. Available: {list(_LITERATURE_DEFAULTS)}")

    entry = _LITERATURE_DEFAULTS[name]

    print(f"\n{'='*62}")
    print(f"  LITERATURE DEFAULTS — {name}")
    print(f"{'='*62}")
    print(f"  Notes    : {entry['notes']}")
    print(f"\n  pop_size : {entry['pop_size']}")
    print(f"  Parameters:")
    if entry['params']:
        for k, v in entry['params'].items():
            print(f"    {k:<10}  {v}")
    else:
        print(f"    (none — algorithm has no internal hyperparameters)")
    print()

    return {
        'params':   dict(entry['params']),
        'pop_size': entry['pop_size'],
        'notes':    entry['notes'],
    }

def getOptimizationAlgorithm(name, n_generations, pop_size, algo_params=None):
    name = name.upper()
    p    = algo_params or {}

    if name == 'AIW_PSO':
        return AIW_PSO(
            epoch=n_generations,
            pop_size=pop_size,
            c1=p.get('c1', 2.05),
            c2=p.get('c2', 2.05),
            alpha=p.get('alpha', 0.4),
        )

    elif name == 'GA':
        return BaseGA(
            epoch=n_generations,
            pop_size=pop_size,
            pc=p.get('pc', 0.9),
            pm=p.get('pm', 0.05),
            selection=p.get('selection', 'tournament'),
            k_way=p.get('k_way', 0.2),
            crossover=p.get('crossover', 'uniform'),
            mutation_multipoints=p.get('mutation_multipoints', True),
            mutation=p.get('mutation', 'flip')
        )

    elif name == 'SADE':
        return SADE(
            epoch=n_generations,
            pop_size=pop_size,
        )

    elif name == 'BSA':
        return OriginalBSA(
            epoch=n_generations,
            pop_size=pop_size,
            ff=p.get('ff', 10),
            pff=p.get('pff', 0.8),
            c_couples=[p.get('c1', 2.0), p.get('c2', 2.0)],
            a_couples=[p.get('a1', 1.5), p.get('a2', 1.5)],
            fc=p.get('fc', 0.5)
        )

    elif name == 'GWO':
        return OriginalGWO(
            epoch=n_generations,
            pop_size=pop_size
        )

    else:
        available = list(ALGORITHM_INFO.keys())
        raise ValueError(f"Algorithm '{name}' not available. Available: {available}")

# 4.1 - Set up Worker

In [7]:
%%writefile worker.py


from __future__ import annotations

import multiprocessing as mp
import os
import numpy as np


def _getActivationLayer(activation):
    from keras import layers
    if activation == 'leaky_relu':
        return layers.LeakyReLU()
    return layers.Activation(activation)


def _getOptimizer(name, learning_rate):
    from keras.optimizers import SGD, AdamW, RMSprop
    name = name.lower()
    if name == 'sgd':
        return SGD(learning_rate=learning_rate, nesterov=True)
    if name == 'rmsprop':
        return RMSprop(learning_rate=learning_rate)
    return AdamW(learning_rate=learning_rate)


def buildTemplateCNN(config, input_shape, num_classes):
    # Imports are function-local so merely importing `worker` in the
    # parent does not drag TensorFlow/Keras into parent memory.
    from keras import layers, models

    k       = config['kernel_size']
    pool_fn = layers.AveragePooling2D if config['pooling'] == 'average' else layers.MaxPooling2D
    base    = config['base_filters']

    # VGG doubling rule — design principle, NOT a search dimension.
    # With n_blocks=3 and base=64 -> [64, 128, 256].
    n_blocks = config['n_blocks']
    filters  = [base * (2 ** i) for i in range(n_blocks)]

    model = models.Sequential(name='Template_CNN')
    model.add(layers.Input(shape=input_shape))

    # Batch normalization is ALWAYS applied after every convolution:
    # design principle, not a search dimension (Ioffe & Szegedy, 2015;
    # Santurkar et al., 2018). See FIXED_BN in optimizer.py for the
    # methodological justification.
    for n_filters in filters:
        # Double conv per block (VGG-style)
        model.add(layers.Conv2D(n_filters, (k, k), padding='same'))
        model.add(layers.BatchNormalization())
        model.add(_getActivationLayer(config['activation']))

        model.add(layers.Conv2D(n_filters, (k, k), padding='same'))
        model.add(layers.BatchNormalization())
        model.add(_getActivationLayer(config['activation']))

        model.add(pool_fn((2, 2)))

    # Classifier head (also topologically fixed)
    model.add(layers.Flatten())
    model.add(layers.Dense(config['dense_units']))
    model.add(_getActivationLayer(config['activation']))
    if config['dropout_rate'] > 0:
        model.add(layers.Dropout(config['dropout_rate']))
    model.add(layers.Dense(num_classes, activation='softmax'))

    model.compile(
        optimizer=_getOptimizer(
            config['optimizer'],
            config['learning_rate'],
        ),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


# ─────────────────────────────────────────────────────────────────────────────
# Dataset serialisation (called once in the parent)
# ─────────────────────────────────────────────────────────────────────────────

def prepareDataFile(data, dir_path: str = '/tmp/hpo_data') -> dict:
    os.makedirs(dir_path, exist_ok=True)
    paths = {
        'x_train':     os.path.join(dir_path, 'x_train.npy'),
        'y_train':     os.path.join(dir_path, 'y_train.npy'),
        'x_test':      os.path.join(dir_path, 'x_test.npy'),
        'y_test':      os.path.join(dir_path, 'y_test.npy'),
        'input_shape': tuple(data['input_shape']),
        'num_classes': int(data['num_classes']),
    }
    np.save(paths['x_train'], data['x_train'])
    np.save(paths['y_train'], data['y_train'])
    np.save(paths['x_test'],  data['x_test'])
    np.save(paths['y_test'],  data['y_test'])
    return paths


def cleanupDataFile(paths: dict) -> None:
    for key in ('x_train', 'y_train', 'x_test', 'y_test'):
        path = paths.get(key)
        if path and os.path.exists(path):
            try:
                os.remove(path)
            except OSError:
                pass
    dir_path = os.path.dirname(paths.get('x_train', '') or '')
    if dir_path and os.path.isdir(dir_path):
        try:
            os.rmdir(dir_path)
        except OSError:
            pass


def _childFn(config, data_paths, eval_epochs, seed, queue):
    """Run a single CNN build+fit; return ('status', val_acc) via `queue`."""
    # ── Silence TF / CUDA / absl startup noise ──────────────────────────────
    # These messages ("cuFFT/cuDNN/cuBLAS factory already registered", "XLA
    # service initialized", "Created device", etc.) are emitted from C++
    # *before* absl initialises its Python bindings — the literal warning
    # "All log messages before absl::InitializeLog() is called are written
    # to STDERR" confirms it. That means TF_CPP_MIN_LOG_LEVEL, absl.logging,
    # and tf.get_logger() all arrive too late to catch them.
    #
    # The only reliable fix is to redirect file descriptor 2 (stderr) at
    # the OS level BEFORE importing tensorflow. This also silences any
    # future C++ noise regardless of which library emits it. The trade-off
    # is that genuine C++ stack traces would also disappear — but our
    # Python-level try/except still captures exceptions via the queue,
    # so diagnostic information about eval success/failure is preserved.
    import sys as _sys
    _sys.stderr.flush()
    _devnull_fd = os.open(os.devnull, os.O_WRONLY)
    os.dup2(_devnull_fd, 2)   # redirect OS-level stderr → /dev/null
    os.close(_devnull_fd)

    # Belt-and-braces: also set the env vars so any child-of-child that
    # inspects them behaves quietly, and in case a future TF release
    # honours them before first-stderr-write.
    os.environ['TF_CPP_MIN_LOG_LEVEL']   = '3'   # 0=all, 3=fatal only
    os.environ['TF_ENABLE_ONEDNN_OPTS']  = '0'
    os.environ['GRPC_VERBOSITY']         = 'ERROR'
    os.environ['GLOG_minloglevel']       = '2'
    os.environ['AUTOGRAPH_VERBOSITY']    = '0'

    # Deterministic per-child seeding (still before any TF import).
    os.environ['PYTHONHASHSEED'] = str(seed)
    import random
    random.seed(seed)
    np.random.seed(seed)

    # Python-side absl logger — handles the W0000/E0000/I0000 lines.
    try:
        import logging
        import absl.logging
        absl.logging.set_verbosity(absl.logging.ERROR)
        logging.getLogger('tensorflow').setLevel(logging.ERROR)
    except Exception:
        pass

    # Import TF inside the child — parent never touches it.
    import tensorflow as tf
    tf.get_logger().setLevel('ERROR')
    tf.autograph.set_verbosity(0)
    tf.random.set_seed(seed)
    tf.config.optimizer.set_jit(False)
    for gpu in tf.config.list_physical_devices('GPU'):
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            # Already configured — harmless.
            pass

    try:
        # Load preprocessed arrays from disk. Linux page cache makes
        # subsequent loads effectively free across sibling processes.
        x_train     = np.load(data_paths['x_train'])
        y_train     = np.load(data_paths['y_train'])
        x_test      = np.load(data_paths['x_test'])
        y_test      = np.load(data_paths['y_test'])
        input_shape = tuple(data_paths['input_shape'])
        num_classes = int(data_paths['num_classes'])

        # `buildTemplateCNN` is defined in *this* module (above),
        # so no external import is required. This is what makes the
        # file self-contained / drop-in on Kaggle.
        model = buildTemplateCNN(config, input_shape, num_classes)

        history = model.fit(
            x_train, y_train,
            validation_data=(x_test, y_test),
            epochs=eval_epochs,
            batch_size=config['batch_size'],
            verbose=0,
        )
        val_acc = float(max(history.history['val_accuracy']))
        queue.put(('ok', val_acc))

    except tf.errors.ResourceExhaustedError as e:
        # GPU OOM — config is too big for the available VRAM.
        queue.put(('oom', str(e)[:300]))

    except Exception as e:
        # Any other error (graph build failure, NaN loss, etc.).
        queue.put(('error', f"{type(e).__name__}: {str(e)[:300]}"))


# ─────────────────────────────────────────────────────────────────────────────
# Parent-side API
# ─────────────────────────────────────────────────────────────────────────────

def evalWorker(
    config: dict,
    data_paths: dict,
    eval_epochs: int = 1,
    timeout: int = 180,
    seed: int = 42,
    start_method: str = 'spawn',
) -> tuple:
    ctx   = mp.get_context(start_method)
    queue = ctx.Queue()
    proc  = ctx.Process(
        target=_childFn,
        args=(config, data_paths, eval_epochs, seed, queue),
    )
    proc.start()
    proc.join(timeout=timeout)

    # Timeout path: child still running when we expected it to finish.
    if proc.is_alive():
        proc.terminate()
        proc.join(timeout=5)
        if proc.is_alive():
            proc.kill()
            proc.join()
        _drainQueue(queue)
        return ('timeout', f'child exceeded {timeout}s')

    # Normal path: child exited; check whether it produced a result.
    try:
        if not queue.empty():
            return queue.get_nowait()
        return ('error', f'child exited (code={proc.exitcode}) with no result')
    finally:
        _drainQueue(queue)


def _drainQueue(q) -> None:
    """Empty a queue silently — avoids resource warnings on close."""
    try:
        while not q.empty():
            try:
                q.get_nowait()
            except Exception:
                break
    except Exception:
        pass
    try:
        q.close()
        q.join_thread()
    except Exception:
        pass


Overwriting worker.py


In [8]:
import sys
sys.path.insert(0, '/kaggle/working')
from worker import prepareDataFile, evalWorker, cleanupDataFile

# 4 - Otimization

In [9]:
PARAM_NAMES = ['batch_idx', 'filters_idx', 'n_blocks_idx', 'dense_idx', 'optimizer_idx', 'pooling_idx', 'kernel_idx']

FIXED_LR         = 1e-3
FIXED_DROPOUT    = 0.5
FIXED_ACTIVATION = 'relu'
FIXED_BN         = True

BOUNDS = [
    IntegerVar(lb=0,    ub=2   ),   # [0]  batch_idx      — 3 options : [32, 64, 128]
    IntegerVar(lb=0,    ub=2   ),   # [1]  filters_idx    — 3 options : [32, 64, 128] (base; doubles per block)
    IntegerVar(lb=0,    ub=1   ),   # [2]  n_blocks_idx   — 2 options : [2, 3]
    IntegerVar(lb=0,    ub=2   ),   # [3]  dense_idx      — 3 options : [128, 256, 512]
    IntegerVar(lb=0,    ub=2   ),   # [4]  optimizer_idx  — 3 options : adamw, sgd, rmsprop
    IntegerVar(lb=0,    ub=1   ),   # [5]  pooling_idx    — 2 options : max, average
    IntegerVar(lb=0,    ub=1   ),   # [6]  kernel_idx     — 2 options : 3, 5
]

BATCH_SIZES   = [32, 64, 128]
FILTER_OPTS   = [32, 64, 128]
DENSE_OPTS    = [128, 256, 512]
OPTIMIZERS    = ['adamw', 'sgd', 'rmsprop']
POOLING_TYPES = ['max', 'average']
KERNEL_SIZES  = [3, 5]
N_BLOCKS_OPTS = [2, 3]

def decodeSolution(solution):
    s = solution
    batch_size    = BATCH_SIZES   [int(s[0])]
    base_filters  = FILTER_OPTS   [int(s[1])]
    n_blocks      = N_BLOCKS_OPTS [int(s[2])]
    dense_units   = DENSE_OPTS    [int(s[3])]
    optimizer     = OPTIMIZERS    [int(s[4])]
    pooling       = POOLING_TYPES [int(s[5])]
    kernel_size   = KERNEL_SIZES  [int(s[6])]

    return {
        # Fixed hyperparameters (see FIXED_* constants above).
        'learning_rate': FIXED_LR,
        'dropout_rate':  FIXED_DROPOUT,
        'activation':    FIXED_ACTIVATION,
        'use_bn':        FIXED_BN,
        'batch_size':    batch_size,
        'base_filters':  base_filters,
        'n_blocks':      n_blocks,
        'dense_units':   dense_units,
        'optimizer':     optimizer,
        'pooling':       pooling,
        'kernel_size':   kernel_size,
    }

In [10]:
_eval_counter  = {'n': 0, 'best': 0.0, 'start': None}
_eval_log      = []   # history of all evaluations

In [11]:
def makeObjectiveWorker(
    data,
    eval_epochs = 1,
    timeout     = 180,
    base_seed   = None,
):
    if base_seed is None:
        base_seed = int(np.random.SeedSequence().entropy % (2**31 - 1))
    base_seed = int(base_seed)

    # Serialise the dataset once; every child reads from disk.
    # Linux's page cache makes repeated loads effectively free.
    data_paths = prepareDataFile(data)
    print(f"  [worker] dataset serialised to "
          f"{os.path.dirname(data_paths['x_train'])}")

    def objective(solution):
        config = decodeSolution(solution)

        # Per-evaluation seed for reproducible weight init across
        # reruns of the same mealpy trajectory.
        child_seed = int(base_seed) + int(_eval_counter['n'])

        status, result = evalWorker(
            config,
            data_paths,
            eval_epochs = eval_epochs,
            timeout     = timeout,
            seed        = child_seed,
        )

        if status == 'ok':
            val_acc = float(result)
        else:
            val_acc = 0.0
            tag = {'oom': 'OOM  ', 'timeout': 'TMOUT',
                   'error': 'ERROR'}.get(status, '???  ')
            print(f"  [{tag}] eval={_eval_counter['n']+1} — {result}")

        fitness = 1.0 - val_acc

        # ── Logging (parent-side; TF never touched here) ──────────────
        _eval_counter['n'] += 1
        if val_acc > _eval_counter['best']:
            _eval_counter['best'] = val_acc
        elapsed = time.time() - _eval_counter['start']
        ram_gb  = _hostRamGB()

        _eval_log.append({
            'eval':          _eval_counter['n'],
            'val_accuracy':  round(val_acc,  4),
            'fitness':       round(fitness,  4),
            'learning_rate': config['learning_rate'],
            'dropout_rate':  config['dropout_rate'],
            'batch_size':    config['batch_size'],
            'base_filters':  config['base_filters'],
            'n_blocks':      config['n_blocks'],
            'dense_units':   config['dense_units'],
            'use_bn':        config['use_bn'],
            'optimizer':     config['optimizer'],
            'activation':    config['activation'],
            'pooling':       config['pooling'],
            'kernel_size':   config['kernel_size'],
            'elapsed_s':     round(elapsed, 1),
            'ram_gb':        round(ram_gb,   3),
        })

        n    = _eval_counter['n']
        best = _eval_counter['best']
        prev_elapsed = _eval_log[-2]['elapsed_s'] if len(_eval_log) >= 2 else 0.0
        dt = elapsed - prev_elapsed
        print(f"  [{n:>4}] val_acc={val_acc:.4f}  best={best:.4f}  "
              f"batch_size={config['batch_size']}  "
              f"base_filters={config['base_filters']}  "
              f"n_blocks={config['n_blocks']}  "
              f"dense_units={config['dense_units']}  "
              f"use_bn={config['use_bn']}  "
              f"optimizer={config['optimizer']}  "
              f"pool={config['pooling']}  "
              f"k={config['kernel_size']}  dt={dt:.0f}s  t={elapsed:.0f}s  "
              f"ram={ram_gb:.2f}GB  [{status}]")

        return fitness

    return objective

In [12]:
def makeObjective(data, eval_epochs=5):
    def objective(solution):
        config = decodeSolution(solution)
        model  = None

        try:
            model = buildTemplateCNN(
                config,
                data['input_shape'],
                data['num_classes']
            )
            history = model.fit(
                data['x_train'], data['y_train'],
                validation_data=(data['x_test'], data['y_test']),
                epochs=eval_epochs,
                batch_size=config['batch_size'],
                verbose=0
            )
            val_acc = max(history.history['val_accuracy'])
            
        except tf.errors.ResourceExhaustedError:
            print(f"  [OOM  ] eval={_eval_counter['n']+1} — config exceeds GPU memory; penalised (val_acc=0)")
            val_acc = 0.0
            oom = True
        except Exception as e:
            print(f"  [ERROR] eval={_eval_counter['n']+1} — {type(e).__name__}: {e}")
            val_acc = 0.0

        try:
            if model is not None and hasattr(model, 'optimizer'):
                # Drop the optimizer FIRST so its slot-variables (Adam's
                # m, v tensors) release their refs before model teardown.
                model.optimizer = None
        except Exception:
            pass
            
        try:
            del model
            del history
        except NameError:
            pass
        
        keras.backend.clear_session()
        gc.collect()
        gc.collect()   # second pass: reclaims objects broken in first pass

        # Sample host RAM AFTER cleanup — this is the level that persists
        # between evals and is the signal for leak detection.
        ram_gb = _hostRamGB()
        fitness = 1.0 - val_acc

        # Logging
        _eval_counter['n'] += 1
        if val_acc > _eval_counter['best']:
            _eval_counter['best'] = val_acc
        elapsed = time.time() - _eval_counter['start']

        _eval_log.append({
            'eval':          _eval_counter['n'],
            'val_accuracy':  round(val_acc,  4),
            'fitness':       round(fitness,  4),
            'learning_rate': config['learning_rate'],
            'dropout_rate':  config['dropout_rate'],
            'batch_size':    config['batch_size'],
            'base_filters':  config['base_filters'],
            'n_blocks':      config['n_blocks'],
            'dense_units':   config['dense_units'],
            'use_bn':        config['use_bn'],
            'optimizer':     config['optimizer'],
            'activation':    config['activation'],
            'pooling':       config['pooling'],
            'kernel_size':   config['kernel_size'],
            'elapsed_s':     round(elapsed, 1),
            'ram_gb':        round(ram_gb,   3),
        })

        n    = _eval_counter['n']
        best = _eval_counter['best']
        # time of this specific evaluation (delta from previous)
        prev_elapsed = _eval_log[-2]['elapsed_s'] if len(_eval_log) >= 2 else 0.0
        dt = elapsed - prev_elapsed
        print(f"  [{n:>4}] val_acc={val_acc:.4f}  best={best:.4f}  "
              f"batch_size={config['batch_size']}  "
              f"base_filters={config['base_filters']}  "
              f"n_blocks={config['n_blocks']}  "
              f"dense_units={config['dense_units']}  "
              f"use_bn={config['use_bn']}  " 
              f"optimizer={config['optimizer']}  "
              f"pool={config['pooling']}  "
              f"k={config['kernel_size']}  dt={dt:.0f}s  t={elapsed:.0f}s  "
              f"ram={ram_gb:.2f}GB")
        
        return fitness

    return objective

In [13]:
def runOptimization(
    algorithm,
    data,
    eval_epochs  = 2,
    n_generations= 20,
    pop_size     = 25,
    results_dir  = 'results/optimization',
    seed         = None,
    algo_params  = None,
    extra_info   = True,
    use_worker   = False,
    worker_timeout = 180,
):
    
    if seed is None:
        seed = int(np.random.SeedSequence().entropy % (2**31 - 1))
    seed = int(seed)

    np.random.seed(seed)
    tf.random.set_seed(seed)

    # Reset global log
    _eval_counter['n']     = 0
    _eval_counter['best']  = 0.0
    _eval_counter['start'] = time.time()
    _eval_log.clear()

    alg_name = algorithm.upper()
    # +1 for the population initialization phase (mealpy evaluates it before iterations)
    total_evals = (n_generations + 1) * pop_size

    print(f"\n{'='*62}")
    print(f"  OPTIMIZATION: {alg_name}")
    print(f"  Generations: {n_generations}  |  Population: {pop_size}  |  "
          f"Total evals: {total_evals}  (init: {pop_size}  +  generations: {n_generations * pop_size})")
    print(f"  Epochs per evaluation: {eval_epochs}")
    print(f"  Search space dimensions: {len(BOUNDS)}  "
          f"(batch, base_filters, n_blocks, dense, opt, pool, kernel)")
    print(f"  Fixed: lr={FIXED_LR}  dropout={FIXED_DROPOUT}  activation={FIXED_ACTIVATION}")
    print(f"  Evaluation mode: "
          f"{'subprocess-isolated (worker)' if use_worker else 'in-process'}")
    print(f"  Seed:            {seed}")
    print(f"{'='*62}\n")

    # Objective function — subprocess-isolated or in-process.
    if use_worker:
        objective = makeObjectiveWorker(
            data,
            eval_epochs = eval_epochs,
            timeout     = worker_timeout,
            base_seed   = seed,
        )
    else:
        objective = makeObjective(
            data,
            eval_epochs = eval_epochs,
        )

    # Problem for mealpy v3
    problem = {
        "obj_func": objective,
        "bounds":   BOUNDS,
        "minmax":   "min",
        "log_to":   None,   # disables mealpy's internal log
    }

    # Algorithm
    algo = getOptimizationAlgorithm(alg_name, n_generations, pop_size, algo_params)

    # Run
    t_start = time.time()
    g_best = algo.solve(problem, seed=seed)
    t_total = time.time() - t_start

    # Best solution (mealpy v3 API)
    best_solution = g_best.solution
    best_fitness  = g_best.target.fitness
    best_config   = decodeSolution(best_solution)
    best_val_acc  = 1.0 - best_fitness

    if extra_info:
        print(f"\n{'='*62}")
        print(f"  FINAL RESULT — {alg_name}")
        print(f"{'='*62}")
        print(f"  Best val_accuracy : {best_val_acc:.4f}  ({best_val_acc*100:.2f}%)")
        print(f"  Total time        : {t_total:.0f}s  ({t_total/60:.1f} min)")
        print(f"  Evaluations done  : {_eval_counter['n']}")
        print(f"\n  Best configuration found:")
        for k, v in best_config.items():
            print(f"    {k:<16} {v}")
        print()

    # Save results
    os.makedirs(results_dir, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')

    run_results = {
        'algorithm':    alg_name,
        'timestamp':    datetime.now().isoformat(),
        'config': {
            'eval_epochs': eval_epochs,
            'n_generations': n_generations,
            'pop_size':    pop_size,
            'seed':        seed,
        },
        'best': {
            'config':       best_config,
            'val_accuracy': round(best_val_acc, 4),
            'fitness':      round(best_fitness, 4),
        },
        'total_time_s':  round(t_total, 1),
        'n_evaluations': _eval_counter['n'],
        'eval_log':      _eval_log.copy(),
    }

    # JSON
    json_path = os.path.join(results_dir, f'{alg_name}_{ts}.json')
    with open(json_path, 'w') as f:
        json.dump(run_results, f, indent=2)
    print()
    print(f"  Results saved : {json_path}")
    
    # CSV of evaluation log
    df_log = pd.DataFrame(_eval_log)
    csv_path = os.path.join(results_dir, f'{alg_name}_{ts}_evals.csv')
    df_log.to_csv(csv_path, index=False)
    print(f"  Eval log      : {csv_path}")
    print()

    return best_config, run_results


# 5 - IRace - Iterated Racing for Automatic Algorithm Configuration


In [14]:
def _suggestParams(trial, algo_name):
    if algo_name == 'AIW_PSO':
        return {
            'c1':    trial.suggest_float('c1',    1.0, 3.0),
            'c2':    trial.suggest_float('c2',    1.0, 3.0),
            'alpha': trial.suggest_float('alpha', 0.0, 1.0),
        }
    
    elif algo_name == 'GA':
        mutation_multipoints = trial.suggest_categorical('mutation_multipoints', [True, False])
        if mutation_multipoints:
            mutation_choices = ['flip', 'swap']
        else:
            mutation_choices = ['flip', 'swap', 'scramble', 'inversion']
        return {
            'pc':                   trial.suggest_float('pc', 0.6, 1.0),
            'pm':                   trial.suggest_float('pm', 0.001, 0.1),
            'selection':            trial.suggest_categorical('selection', ['roulette', 'tournament', 'random']),
            'k_way':                trial.suggest_float('k_way', 0.1, 0.5),
            'crossover':            trial.suggest_categorical('crossover', ['one_point', 'multi_points', 'uniform', 'arithmetic']),
            'mutation_multipoints': mutation_multipoints,
            'mutation':             trial.suggest_categorical('mutation', mutation_choices),
        }
    elif algo_name == 'BSA':
        return {
            'ff':  trial.suggest_int  ('ff',  5,   20),
            'pff': trial.suggest_float('pff', 0.1,  1.0),
            'c1':  trial.suggest_float('c1',  0.5,  3.0),
            'c2':  trial.suggest_float('c2',  0.5,  3.0),
            'a1':  trial.suggest_float('a1',  0.5,  3.0),
            'a2':  trial.suggest_float('a2',  0.5,  3.0),
            'fc':  trial.suggest_float('fc',  0.1,  1.0),
        }

    elif algo_name == 'SADE':
        # SADE is fully self-adaptive: F, CR, and the mutation strategy are
        # updated every iteration from the success history of previous trials.
        # Nothing for Optuna to tune here.
        return {}

    elif algo_name == 'GWO':
        # Grey Wolf Optimizer has no behavioural hyperparameters exposed.
        return {}

    else:
        return {}


In [15]:
def configureOptimizationAlgorithmIRACE(
    algorithm,
    data,
    n_trials        = 15,
    n_generations   = 3,
    pop_size        = 10,
    eval_epochs     = 2,
    n_repeat_trials = 1,
    results_dir     = 'results/irace',
    seed            = 42,
    extra_info      = True,
    save_plots      = True,
):
    alg_name       = algorithm.upper()
    fixed_pop_size = pop_size

    os.makedirs(results_dir, exist_ok=True)
    ts         = datetime.now().strftime('%Y%m%d_%H%M%S')
    study_name = f'irace_{alg_name}_{ts}'

    print(f"\n{'='*62}")
    print(f"  IRACE - Configuring {alg_name}")
    print(f"  Trials: {n_trials}  |  Repetitions/trial: {n_repeat_trials}  |  "
          f"pop_size: {fixed_pop_size} (fixed)  |  n_generations: {n_generations}  |  epochs: {eval_epochs}")
    print(f"  Study name: {study_name}")
    print(f"{'='*62}\n")

    t_start = time.time()

    def optunaObjective(trial):
        algo_params = _suggestParams(trial, alg_name)

        params_str = '  '.join(
            f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}"
            for k, v in algo_params.items()
        )
        print(f"  [Trial {trial.number+1:>3}/{n_trials}]  pop_size={fixed_pop_size}  {params_str}")
        scores = []
        for i in range(n_repeat_trials):
            _, run_results = runOptimization(
                alg_name,
                data,
                eval_epochs   = eval_epochs,
                n_generations = n_generations,
                pop_size      = fixed_pop_size,
                results_dir   = os.path.join(results_dir, 'races'),
                seed          = seed + i,   # BLOCKING: identical seed set per trial
                algo_params   = algo_params,
                extra_info    = extra_info,
                use_worker    = True,
                worker_timeout= 180,
            )
            scores.append(run_results['best']['val_accuracy'])

        return float(np.mean(scores))

    study = optuna.create_study(
        study_name = study_name,
        direction  = 'maximize',
        sampler    = optuna.samplers.TPESampler(
            seed         = seed,
            multivariate = True,   # learns correlations between parameters
            group        = True,   # handles conditional parameters (e.g. mutation)
        ),
    )
    study.optimize(optunaObjective, n_trials=n_trials, show_progress_bar=False)

    t_total = time.time() - t_start
    best_params = study.best_params
    best_value  = study.best_value

    print(f"\n{'='*62}")
    print(f"  IRACE RESULT - {alg_name}")
    print(f"{'='*62}")
    print(f"  Best val_accuracy (race) : {best_value:.4f}")
    print(f"  Configuration time       : {t_total:.0f}s  ({t_total/60:.1f} min)")
    print(f"  Fixed pop_size           : {fixed_pop_size}")
    print(f"  Best parameters found:")
    for k, v in best_params.items():
        v_str = f"{v:.4f}" if isinstance(v, float) else str(v)
        print(f"    {k:<24}  {v_str}")
    print()

    # ── Save standard Optuna plots as PNG ──────────────────────────────
    # Uses optuna.visualization.matplotlib (no kaleido dependency).
    plot_paths = {}
    if save_plots:
        plots_dir = os.path.join(results_dir, f'{study_name}_plots')
        os.makedirs(plots_dir, exist_ok=True)

        plot_specs = [
            ('optimization_history', lambda: ovm.plot_optimization_history(study)),
            ('param_importances',    lambda: ovm.plot_param_importances(study)),
            ('parallel_coordinate',  lambda: ovm.plot_parallel_coordinate(study)),
            ('slice',                lambda: ovm.plot_slice(study)),
            ('edf',                  lambda: ovm.plot_edf(study)),
            # plot_contour is O(n_params^2) — skip for >6 params to stay fast.
            *([('contour', lambda: ovm.plot_contour(study))]
              if len(study.best_params) <= 6 else []),
        ]

        for name, maker in plot_specs:
            try:
                ax = maker()   # returns a matplotlib Axes (or array of Axes)
                fig = (ax.figure if hasattr(ax, 'figure')
                       else ax.flatten()[0].figure)
                fig.set_size_inches(max(8, fig.get_size_inches()[0]),
                                    max(5, fig.get_size_inches()[1]))
                fig.tight_layout()
                out_path = os.path.join(plots_dir, f'{name}.png')
                fig.savefig(out_path, dpi=150, bbox_inches='tight')
                plt.close(fig)
                plot_paths[name] = out_path
                print(f"  Plot saved: {out_path}")
            except Exception as e:
                print(f"  [WARN] Could not generate plot '{name}': "
                      f"{type(e).__name__}: {e}")

    # ── Save metadata JSON ─────────────────────────────────────────────
    result = {
        'algorithm':    alg_name,
        'timestamp':    datetime.now().isoformat(),
        'study_name':   study_name,
        'n_trials':     n_trials,
        'race_budget':  {'n_generations': n_generations, 'pop_size': fixed_pop_size, 'eval_epochs': eval_epochs},
        'best_params':  best_params,
        'best_val_accuracy_race': round(best_value, 4),
        'config_time_s': round(t_total, 1),
        'plot_paths':    plot_paths,
        'all_trials': [
            {
                'trial':        t.number,
                'params':       t.params,
                'val_accuracy': t.value,
            }
            for t in study.trials
        ],
    }
    json_path = os.path.join(results_dir, f'irace_{alg_name}_{ts}.json')
    with open(json_path, 'w') as f:
        json.dump(result, f, indent=2)
    print(f"  Results saved: {json_path}")

    return best_params, study


# 6 - Validation

In [16]:
def analyzeParamImportance(study, algo_name, save_path=None):
    import matplotlib.pyplot as plt
    import numpy as np

    # Try fANOVA first, fall back to tree-based importance
    try:
        importances = get_param_importances(study, evaluator=FanovaImportanceEvaluator())
        method = 'fANOVA (Hutter et al., 2014)'
    except Exception:
        try:
            importances = get_param_importances(study)
            method = 'Mean Decrease Impurity (tree-based)'
        except Exception as e:
            print(f"  [ERROR] Importance analysis unavailable: {e}")
            return {}

    sep = '=' * 62
    print(f"\n{sep}")
    print(f"  PARAMETER IMPORTANCE — {algo_name.upper()}")
    print(f"  Method: {method}")
    print(sep)
    print(f"  {'Parameter':<22}  {'Score':>8}  Bar")
    print(f"  {'-'*54}")
    for param, score in importances.items():
        bar = '█' * max(1, int(score * 35))
        print(f"  {param:<22}  {score:>8.4f}  {bar}")
    print()
    print("  Interpretation:")
    print("    High score → algorithm is sensitive to this parameter on your problem")
    print("    Low score  → parameter has little effect; literature default is acceptable")
    print()

    # Horizontal bar chart
    names  = list(importances.keys())
    scores = list(importances.values())
    max_s  = max(scores) if scores else 1
    colors = ['#e74c3c' if s == max_s else '#3498db' for s in scores]

    fig, ax = plt.subplots(figsize=(max(7, len(names) * 1.3), 4))
    bars = ax.barh(names[::-1], scores[::-1], color=colors[::-1],
                   alpha=0.85, edgecolor='black', linewidth=0.5)
    for bar, score in zip(bars, scores[::-1]):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                f'{score:.3f}', va='center', fontsize=9)
    ax.set_xlabel('Importance Score', fontsize=11)
    ax.set_title(
        f'Parameter Importance — {algo_name.upper()}\n'
        f'({method})',
        fontsize=12, fontweight='bold'
    )
    ax.set_xlim([0, max_s * 1.25])
    ax.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  Importance chart saved: {save_path}")
    plt.show()

    return dict(importances)

In [17]:
def validateConfiguration(
    algo_name,
    tuned_params,
    data,
    baseline_params = None,
    n_runs          = 10,
    n_generations   = 15,
    pop_size        = 25,
    eval_epochs     = 5,
    results_dir     = 'results/irace',
    seed            = 42,
):
    from scipy.stats import mannwhitneyu
    import numpy as np

    algo_name = algo_name.upper()
    os.makedirs(results_dir, exist_ok=True)

    # ── Resolve baseline ──────────────────────────────────────────────────────
    if baseline_params is None:
        lit            = getLiteratureDefaults(algo_name)
        baseline_pop   = lit['pop_size']
        baseline_algo  = lit['params']
        baseline_label = 'Literature defaults'
    else:
        bp             = dict(baseline_params)
        baseline_pop   = bp.pop('pop_size', pop_size)
        baseline_algo  = bp
        baseline_label = 'Custom baseline'

    # ── tuned_params contains only internal algorithm parameters (no pop_size) ─
    tuned_algo = dict(tuned_params)
    tuned_pop  = pop_size

    sep = '=' * 62
    print(f"\n{sep}")
    print(f"  STATISTICAL VALIDATION — {algo_name}")
    print(f"  Runs per configuration : {n_runs}  (independent seeds)")
    print(f"  n_generations={n_generations}  pop_size={pop_size}  eval_epochs={eval_epochs}")
    print(f"  Baseline: {baseline_label}")
    print(sep)

    def _run_config(algo_params, pop_sz, label):
        scores = []
        for i in range(n_runs):
            print(f"  [{label}] run {i+1:>2}/{n_runs} ...", end=' ', flush=True)
            _, run_res = runOptimization(
                algo_name, data,
                eval_epochs  = eval_epochs,
                n_generations  = n_generations,
                pop_size     = pop_sz,
                results_dir  = os.path.join(results_dir, 'validation_races'),
                seed         = seed + i * 100,
                algo_params  = algo_params,
            )
            acc = run_res['best']['val_accuracy']
            scores.append(acc)
            print(f"val_acc={acc:.4f}")
        return scores

    tuned_scores    = _run_config(tuned_algo,   tuned_pop,   'TUNED   ')
    baseline_scores = _run_config(baseline_algo, baseline_pop, 'BASELINE')

    # ── Statistics ────────────────────────────────────────────────────────────
    t = np.array(tuned_scores)
    b = np.array(baseline_scores)

    _, p_value   = mannwhitneyu(t, b, alternative='two-sided')
    significant  = bool(p_value < 0.05)

    if t.mean() > b.mean():
        winner = 'tuned' if significant else 'tie (not significant)'
    elif b.mean() > t.mean():
        winner = 'baseline' if significant else 'tie (not significant)'
    else:
        winner = 'tie'

    print(f"\n{sep}")
    print(f"  RESULTS — {algo_name}")
    print(sep)
    print(f"  {'Config':<14}  {'Mean':>8}  {'Std':>8}  {'Min':>8}  {'Max':>8}")
    print(f"  {'-'*56}")
    print(f"  {'TUNED':<14}  {t.mean():>8.4f}  {t.std():>8.4f}  {t.min():>8.4f}  {t.max():>8.4f}")
    print(f"  {'BASELINE':<14}  {b.mean():>8.4f}  {b.std():>8.4f}  {b.min():>8.4f}  {b.max():>8.4f}")
    print()
    print(f"  Mann-Whitney U p-value       : {p_value:.4f}")
    print(f"  Significant (p < 0.05)       : {'YES ✓' if significant else 'NO'}")
    print(f"  Winner                       : {winner.upper()}")
    print()
    if not significant:
        print("  NOTE: The difference is not statistically significant.")
        print("        Consider increasing n_runs (≥30 for publication) or n_trials.")
    print()

    result = {
        'algorithm':      algo_name,
        'n_runs':         n_runs,
        'baseline_label': baseline_label,
        'tuned': {
            'params':   tuned_algo,
            'pop_size': tuned_pop,
            'scores':   tuned_scores,
            'mean':     float(t.mean()),
            'std':      float(t.std()),
            'min':      float(t.min()),
            'max':      float(t.max()),
        },
        'baseline': {
            'params':   baseline_algo,
            'pop_size': baseline_pop,
            'scores':   baseline_scores,
            'mean':     float(b.mean()),
            'std':      float(b.std()),
            'min':      float(b.min()),
            'max':      float(b.max()),
        },
        'p_value':    float(p_value),
        'significant': significant,
        'winner':      winner,
    }

    # Save JSON
    ts        = datetime.now().strftime('%Y%m%d_%H%M%S')
    json_path = os.path.join(results_dir, f'validation_{algo_name}_{ts}.json')
    with open(json_path, 'w') as f:
        json.dump(result, f, indent=2)
    print(f"  Validation JSON saved: {json_path}")

    # Box plot
    plot_path = os.path.join(results_dir, f'validation_{algo_name}_{ts}.png')
    _plotValidationBoxplot(tuned_scores, baseline_scores, algo_name, p_value, plot_path)

    return result

In [18]:
def configureAndValidate(
    algo_name,
    data,
        
    n_trials         = 15,
    irace_generations= 3,
    irace_pop_size   = 10,
    irace_eval_epochs= 3,

    n_runs           = 10,
    n_generations    = 15,
    eval_epochs      = 5,
    
    results_dir      = 'results/irace',
    seed             = 42,
):
    algo_name = algo_name.upper()
    os.makedirs(results_dir, exist_ok=True)

    # ── 1. Literature defaults ────────────────────────────────────────────────
    lit = getLiteratureDefaults(algo_name)

    # ── 2. IRACE with reduced budget (racing phase) ────────────────────────────
    best_params, study = configureOptimizationAlgorithmIRACE(
        algo_name,
        data,
        n_trials       = n_trials,
        n_generations  = irace_generations,
        pop_size       = irace_pop_size,
        eval_epochs    = irace_eval_epochs,
        results_dir    = results_dir,
        seed           = seed,
    )

    # ── 3. Parameter importance analysis ─────────────────────────────────────
    importance_path = os.path.join(results_dir, f'importance_{algo_name}.png')
    importances     = analyzeParamImportance(study, algo_name, save_path=importance_path)

    # ── 4. Statistical validation ─────────────────────────────────────────────
    # pop_size comes from literature defaults — both tuned and baseline configs
    # run with the same population size to ensure a fair comparison.
    validation = validateConfiguration(
        algo_name,
        dict(best_params),
        data,
        baseline_params = None, # auto-uses literature defaults
        n_runs          = n_runs,
        n_generations   = n_generations,
        pop_size        = lit['pop_size'],
        eval_epochs     = eval_epochs,
        results_dir     = results_dir,
        seed            = seed,
    )

    # ── 5. Save comprehensive report ──────────────────────────────────────────
    ts     = datetime.now().strftime('%Y%m%d_%H%M%S')
    report = {
        'algorithm':  algo_name,
        'timestamp':  datetime.now().isoformat(),
        'literature_defaults': {
            'params':   lit['params'],
            'pop_size': lit['pop_size'],
        },
        'tuning': {
            'n_trials':        n_trials,
            'race_n_generations': irace_generations,
            'race_eval_epochs': irace_eval_epochs,
            'best_params':     best_params,
            'best_val_acc_race': round(study.best_value, 4),
        },
        'importance': importances,
        'validation': {k: v for k, v in validation.items()},
    }
    report_path = os.path.join(results_dir, f'full_report_{algo_name}_{ts}.json')
    with open(report_path, 'w') as f:
        json.dump(report, f, indent=2)

    sep = '=' * 62
    print(f"\n{sep}")
    print(f"  CONFIGURE & VALIDATE COMPLETE — {algo_name}")
    print(sep)
    print(f"  Literature baseline   : {lit['notes'][:60]}...")
    print(f"  Best params (tuned)   : {best_params}")
    print(f"  Most important param  : {next(iter(importances), 'n/a')}")
    print(f"  Validation winner     : {validation['winner'].upper()}")
    print(f"  p-value               : {validation['p_value']:.4f}  "
          f"({'significant' if validation['significant'] else 'not significant'})")
    print(f"\n  Full report saved     : {report_path}")
    print()

    return {
        'literature_defaults': lit,
        'best_params':         best_params,
        'study':               study,
        'importances':         importances,
        'validation':          validation,
    }

# 7 — Optimização Evolutiva

In [19]:
# 1. Carregar o dataset
data_cifar = getDataset('cifar10')
print(f"  x_train : {data_cifar['x_train'].shape}")
print(f"  x_test  : {data_cifar['x_test'].shape}")
print(f"  classes : {data_cifar['class_names']}")

  CIFAR-10 loaded:
    Train      :  50000 samples
    Test       :  10000 samples
    Shape      : (32, 32, 3)
    Classes    : ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
  x_train : (50000, 32, 32, 3)
  x_test  : (10000, 32, 32, 3)
  classes : ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [20]:
#data_mnist = getDataset("mnist")
#print(f"  x_train : {data_mnist['x_train'].shape}")
#print(f"  x_test  : {data_mnist['x_test'].shape}")
#print(f"  classes : {data_mnist['class_names']}")

## SA-DE Algoithm

In [26]:
N_RUNS = 1

for i in range(N_RUNS):
    print('=' * 72)
    print(f'Run {i + 1}/{N_RUNS}')
    print('=' * 72)

    best_config, run = runOptimization(
        algorithm     = 'GA',
        data          = data_cifar,
        n_generations = 10,
        pop_size      = 20,
        eval_epochs   = 1,
            results_dir   = 'results/ga/cifar-10',
        use_worker    = True,
        worker_timeout= 180,
    )

    print(f'[Run {i + 1}] Best val_accuracy :', run['best']['val_accuracy'])
    print(f'[Run {i + 1}] Best config       :', best_config)


Run 1/1

  OPTIMIZATION: GA
  Generations: 10  |  Population: 20  |  Total evals: 220  (init: 20  +  generations: 200)
  Epochs per evaluation: 1
  Search space dimensions: 7  (batch, base_filters, n_blocks, dense, opt, pool, kernel)
  Fixed: lr=0.001  dropout=0.5  activation=relu
  Evaluation mode: subprocess-isolated (worker)
  Seed:            1310872421

  [worker] dataset serialised to /tmp/hpo_data
  [   1] val_acc=0.4383  best=0.4383  batch_size=64  base_filters=64  n_blocks=2  dense_units=256  use_bn=True  optimizer=sgd  pool=max  k=3  dt=26s  t=26s  ram=1.88GB  [ok]
  [   2] val_acc=0.4772  best=0.4772  batch_size=64  base_filters=64  n_blocks=3  dense_units=256  use_bn=True  optimizer=sgd  pool=average  k=5  dt=29s  t=55s  ram=1.88GB  [ok]
  [   3] val_acc=0.4200  best=0.4772  batch_size=32  base_filters=32  n_blocks=2  dense_units=128  use_bn=True  optimizer=adamw  pool=max  k=3  dt=37s  t=92s  ram=1.88GB  [ok]
  [   4] val_acc=0.4401  best=0.4772  batch_size=32  base_filter

## AIW_PSO

In [22]:
#best_params, study = configureOptimizationAlgorithmIRACE(
#        "AIW_PSO",
#        data,
#        n_trials       = 8,
#        n_generations  = 5,
#        pop_size       = 10,
#        eval_epochs    = 1,
#        n_repeat_trials= 1,
#        results_dir    = 'results/irace',
#        seed           = 42,
#    )